# 07 - Fixed-Origin ML vs DCA

In this notebook, we compare simple ML forecasts against the DCA benchmarks using a stricter fixed-origin setup.

The forecast origin is month 24.

That means the model can use production history through month 24 only.

The forecast-check window is months 25-33.

This setup is more directly comparable to the DCA notebooks because both approaches are asked to forecast the same future window from the same information cutoff.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"
DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"
ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

DATA_FILE.exists(), DCA_OUTPUT_DIR.exists(), ML_OUTPUT_DIR.exists()

In [ ]:
df = pd.read_csv(DATA_FILE)

df.head()

In [ ]:
id_columns = [
    "api8",
    "district",
    "lease_no",
    "well_no",
]

date_label_columns = [
    "first_prod_month",
    "cycle_year_month",
    "reported_first_month",
    "first_positive_prod_month",
]

numeric_columns = [
    "month_on_production",
    "oil_bbl",
    "casinghead_gas_mcf",
    "boe",
    "interval_length_proxy_ft",
    "reported_month_on_production",
]

for column in id_columns:
    df[column] = df[column].astype(str)

for column in date_label_columns:
    df[column] = df[column].astype(str)

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes

In [ ]:
df = df.sort_values(["api8", "month_on_production"]).reset_index(drop=True)

df.head()

In [ ]:
cohort_summary = pd.Series(
    {
        "row_count": len(df),
        "well_count": df["api8"].nunique(),
        "first_month_on_production": df["month_on_production"].min(),
        "last_month_on_production": df["month_on_production"].max(),
    }
)

cohort_summary

## Fixed-Origin Forecast Rule

The forecast origin is month 24.

For each well, the model can use information available through month 24 only.

The model will then forecast oil production for months 25-33.

Observed production from months 25-33 must not be used as an input feature.

This is stricter than the rolling one-step-ahead setup in notebook 06 and is more directly comparable to the DCA workflow.

In [ ]:
modeling_df = df[df["month_on_production"].between(1, 24)].copy()

well_groups = modeling_df.groupby("api8", group_keys=False)

modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)

modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]

modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

train_rows = modeling_df.dropna(
    subset=[
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
).copy()

train_rows = train_rows[train_rows["target_month_on_production"].between(13, 24)].copy()

train_rows.shape

In [ ]:
feature_columns = [
    "month_on_production",
    "last_observed_oil_bbl",
    "trailing_3mo_avg_oil_bbl",
    "trailing_6mo_avg_oil_bbl",
    "interval_length_proxy_ft",
]

target_column = "target_next_oil_bbl"

X_train = train_rows[feature_columns]
y_train = train_rows[target_column]

X_train.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

random_forest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)

random_forest_model.fit(X_train, y_train)

In [ ]:
origin_rows = df[df["month_on_production"].between(1, 24)].copy()

origin_groups = origin_rows.groupby("api8", group_keys=False)

origin_rows["trailing_3mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

origin_rows["trailing_6mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

month_24_rows = origin_rows[origin_rows["month_on_production"] == 24].copy()

month_24_rows["last_observed_oil_bbl"] = month_24_rows["oil_bbl"]

month_24_rows[
    [
        "api8",
        "lease_name",
        "well_no",
        "month_on_production",
        "oil_bbl",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].head()

In [ ]:
month_24_rows.shape, month_24_rows["api8"].nunique()

In [ ]:
forecast_months = pd.DataFrame(
    {
        "target_month_on_production": list(range(25, 34))
    }
)

fixed_origin_rows = month_24_rows.merge(
    forecast_months,
    how="cross",
)

fixed_origin_rows[
    [
        "api8",
        "lease_name",
        "well_no",
        "month_on_production",
        "target_month_on_production",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
    ]
].head(12)

In [ ]:
fixed_origin_rows["forecast_month_on_production"] = fixed_origin_rows[
    "target_month_on_production"
]

fixed_origin_rows[
    [
        "api8",
        "month_on_production",
        "forecast_month_on_production",
        "target_month_on_production",
    ]
].head(12)

In [ ]:
fixed_origin_X = fixed_origin_rows[
    [
        "forecast_month_on_production",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].copy()

fixed_origin_X = fixed_origin_X.rename(
    columns={
        "forecast_month_on_production": "month_on_production"
    }
)

fixed_origin_X.head()

In [ ]:
fixed_origin_rows["linear_regression_forecast_oil_bbl"] = linear_model.predict(fixed_origin_X)

fixed_origin_rows["random_forest_forecast_oil_bbl"] = random_forest_model.predict(fixed_origin_X)

fixed_origin_rows[
    [
        "api8",
        "target_month_on_production",
        "linear_regression_forecast_oil_bbl",
        "random_forest_forecast_oil_bbl",
    ]
].head(12)

In [ ]:
actual_rows = df[df["month_on_production"].between(25, 33)][
    [
        "api8",
        "month_on_production",
        "oil_bbl",
    ]
].copy()

actual_rows = actual_rows.rename(
    columns={
        "month_on_production": "target_month_on_production",
        "oil_bbl": "actual_oil_bbl",
    }
)

fixed_origin_rows = fixed_origin_rows.merge(
    actual_rows,
    on=["api8", "target_month_on_production"],
    how="left",
)

fixed_origin_rows[
    [
        "api8",
        "target_month_on_production",
        "actual_oil_bbl",
        "linear_regression_forecast_oil_bbl",
        "random_forest_forecast_oil_bbl",
    ]
].head(12)

In [ ]:
fixed_origin_rows["naive_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "last_observed_oil_bbl"
]

fixed_origin_rows["trailing_3mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_3mo_avg_oil_bbl"
]

fixed_origin_rows["trailing_6mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_6mo_avg_oil_bbl"
]

fixed_origin_rows[
    [
        "api8",
        "target_month_on_production",
        "actual_oil_bbl",
        "naive_fixed_origin_forecast_oil_bbl",
        "trailing_3mo_fixed_origin_forecast_oil_bbl",
        "trailing_6mo_fixed_origin_forecast_oil_bbl",
        "linear_regression_forecast_oil_bbl",
        "random_forest_forecast_oil_bbl",
    ]
].head(12)


In [ ]:
def summarize_forecast_method(results_df, forecast_column, model_name):
    error = results_df[forecast_column] - results_df["actual_oil_bbl"]
    absolute_error = error.abs()
    
    return {
        "model": model_name,
        "mae_bbl": absolute_error.mean(),
        "bias_bbl": error.mean(),
        "wape": absolute_error.sum() / results_df["actual_oil_bbl"].sum(),
    }

In [ ]:
fixed_origin_metrics = pd.DataFrame(
    [
        summarize_forecast_method(
            fixed_origin_rows,
            "naive_fixed_origin_forecast_oil_bbl",
            "naive_fixed_origin",
        ),
        summarize_forecast_method(
            fixed_origin_rows,
            "trailing_3mo_fixed_origin_forecast_oil_bbl",
            "trailing_3mo_fixed_origin",
        ),
        summarize_forecast_method(
            fixed_origin_rows,
            "trailing_6mo_fixed_origin_forecast_oil_bbl",
            "trailing_6mo_fixed_origin",
        ),
        summarize_forecast_method(
            fixed_origin_rows,
            "linear_regression_forecast_oil_bbl",
            "linear_regression_fixed_origin",
        ),
        summarize_forecast_method(
            fixed_origin_rows,
            "random_forest_forecast_oil_bbl",
            "random_forest_fixed_origin",
        ),
    ]
)

fixed_origin_metrics.sort_values("mae_bbl").round(2)

In [ ]:
fixed_origin_metrics_display = fixed_origin_metrics.copy()

fixed_origin_metrics_display["wape_percent"] = fixed_origin_metrics_display["wape"] * 100

fixed_origin_metrics_display = fixed_origin_metrics_display.sort_values("mae_bbl").reset_index(drop=True)

fixed_origin_metrics_display.round(2)

In [ ]:
ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fixed_origin_metrics_display.to_csv(
    ML_OUTPUT_DIR / "fixed_origin_ml_vs_baseline_metrics.csv",
    index=False,
)

In [ ]:
fixed_origin_output_columns = [
    "api8",
    "lease_name",
    "well_no",
    "target_month_on_production",
    "actual_oil_bbl",
    "naive_fixed_origin_forecast_oil_bbl",
    "trailing_3mo_fixed_origin_forecast_oil_bbl",
    "trailing_6mo_fixed_origin_forecast_oil_bbl",
    "linear_regression_forecast_oil_bbl",
    "random_forest_forecast_oil_bbl",
]

fixed_origin_rows[fixed_origin_output_columns].to_csv(
    ML_OUTPUT_DIR / "fixed_origin_ml_vs_baseline_predictions.csv",
    index=False,
)

In [ ]:
sorted(path.name for path in DCA_OUTPUT_DIR.glob("*.csv"))

In [ ]:
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "dca_workflow" / "outputs"

NOTEBOOK_OUTPUT_DIR.exists(), sorted(path.name for path in NOTEBOOK_OUTPUT_DIR.glob("*.csv"))

In [ ]:
report_output_folders = {
    "dca_outputs": DCA_OUTPUT_DIR,
    "ml_outputs": ML_OUTPUT_DIR,
}

for folder_name, folder_path in report_output_folders.items():
    print(folder_name)
    print(folder_path)
    print(sorted(path.name for path in folder_path.glob("*.csv")))
    print()

In [ ]:
all_csv_files = sorted(path for path in PROJECT_ROOT.rglob("*.csv"))

[
    str(path.relative_to(PROJECT_ROOT))
    for path in all_csv_files
    if any(
        keyword in path.name.lower()
        for keyword in ["per_well", "comparison", "harmonic", "exponential", "hyperbolic"]
    )
]

## DCA Comparison File Check

The fixed-origin ML outputs were created successfully under `reports/ml_outputs`.

However, the repo does not currently contain a per-well DCA forecast comparison CSV with months 25-33 forecast totals.

The available files under `reports/dca_outputs` are summary, median-curve, and fit-summary outputs.

Because the per-well DCA forecast totals are not available as a saved CSV in this repo state, this notebook will not force a direct ML-vs-DCA merge yet.

The next project step is to export a DCA comparison-ready file with, at minimum:

- `api8`
- actual oil over months 25-33
- exponential DCA forecast oil over months 25-33
- hyperbolic DCA forecast oil over months 25-33
- harmonic DCA forecast oil over months 25-33, if available

## Fixed-Origin ML Result

This notebook created fixed-origin forecasts from month 24 for months 25-33.

In this setup, each well's production-history features were frozen at month 24.

The model was allowed to know which future production month it was forecasting, but it was not allowed to use observed oil from months 25-33 as input.

The best fixed-origin method in this notebook was `[BEST_MODEL]`.

Its MAE was approximately `[MAE_VALUE]` barrels per forecasted month.

Its WAPE was approximately `[WAPE_PERCENT]%`.

This is a stricter test than the rolling one-step-ahead setup in notebook 06.

The direct comparison against DCA is still pending because the repo does not currently include a per-well DCA forecast-total CSV for months 25-33.

## QA Checklist

Before using this notebook in the project summary:

- Confirm the notebook runs from top to bottom without errors
- Confirm the forecast origin is month 24
- Confirm forecast target months are 25-33
- Confirm production-history features are frozen at month 24
- Confirm observed oil from months 25-33 is used only for scoring, not as input features
- Confirm fixed-origin ML outputs were saved under `reports/ml_outputs`
- Confirm the missing DCA per-well comparison file is documented

In [ ]:
sorted(path.name for path in ML_OUTPUT_DIR.glob("fixed_origin*.csv"))

## Notebook Status

This notebook is complete as a fixed-origin ML baseline workflow.

It creates month-24-origin forecasts for months 25-33, saves forecast-level and metrics-level ML outputs, and documents why the direct DCA merge is pending.

The next required artifact for full ML-vs-DCA comparison is a per-well DCA forecast-total CSV for months 25-33.